In [ ]:
# Labeling script: uses the already trained, best-performing DanskBERT model
# (saved in ./final_model_exp from the training notebook) to label a
# new, unlabeled dataset of approximately 160,000 Facebook comments for
# explicit hate speech/non-hate speech (exp/no-hs).

In [ ]:
%pip install --no-cache-dir \
    "scikit-learn==1.9.0" \
    "datasets==5.0.1" \
    "evaluate" \
    "accelerate" \
    "torch==2.13.0" \
    "tokenizers" \
    "sentencepiece" \
    "transformers==4.57.6" \
    "huggingface_hub==0.36.0"

In [2]:
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
)
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

In [ ]:
# Configuration

INPUT_CSV = "all_unlabelled.csv"    # dataset of unlabeled comments
TEXT_COL = "comment_text"
OUTPUT_CSV = "all_labelled_exp.csv"
MODEL_DIR = "./final_model_exp"      
BATCH_SIZE = 64                    

In [ ]:
# Loading data

df = pd.read_csv(INPUT_CSV, encoding="utf-8-sig")
print("Shape:", df.shape)

In [ ]:
# Text pre-processing (same as in training script: removal of URLs and redundant white space)

df["comment_text_clean"] = (
    df[TEXT_COL]
    .astype(str)
    .str.replace(r"https?://\S+|www\.\S+", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)
 

In [ ]:
# Unpacks zipped folder with final model

import shutil

shutil.unpack_archive("final_model_exp.zip", "./final_model_exp")

import os
print(os.listdir("./final_model_exp"))

In [ ]:
# Loading fine-tuned model and tokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
 
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, local_files_only=True)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_DIR, local_files_only=True
)
model.to(device)
model.eval()  

In [ ]:
# Converting to huggingface dataset and tokenize

ds = Dataset.from_pandas(df[["comment_text_clean"]], preserve_index=False)
 
def tokenize(batch):
    return tokenizer(
        batch["comment_text_clean"],
        truncation=True,
        max_length=256,
        padding=False,  # Dynamic padding handled by DataCollatorWithPadding.
    )
 
ds = ds.map(tokenize, batched=True)
ds.set_format("torch", columns=["input_ids", "attention_mask"])
 
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=data_collator)

In [ ]:
# Run inferece on all comments

all_preds = []
all_probs_exp = []  # probability of class 1 (exp)
 
with torch.no_grad():
    for batch in tqdm(loader, desc="Labeling comments"):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        probs = torch.softmax(outputs.logits, dim=-1)
        preds = torch.argmax(probs, dim=-1)
 
        all_preds.extend(preds.cpu().numpy().tolist())
        all_probs_exp.extend(probs[:, 1].cpu().numpy().tolist())

In [ ]:
# Collect results and save to CSV

df["exp_pred"] = all_preds     # 0 = no-hs, 1 = exp
df["exp_prob"] = all_probs_exp  # the model's probability (0–1) of explicit hate speech
 
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"Done! Saved {len(df)} labeled rows to {OUTPUT_CSV}")
print(df["exp_pred"].value_counts())